# Pelatihan M1 (ResNet50) dan M2 (ResNet50-GLCM)
Notebook ini melatih **baseline M1** dan **model usulan M2** sesuai alur proyek.

**Cara pakai di VS Code:**
1. Buka folder proyek di VS Code.
2. Pilih kernel Python / virtual environment (`pip install -r requirements.txt`).
3. Jalankan sel satu per satu secara berurutan.

In [ ]:
import os
import tensorflow as tf

n_kelas = len([d for d in os.listdir('dataset') if os.path.isdir(f'dataset/{d}')])
print('Jumlah kelas dataset:', n_kelas)
print('Status eda_output:', 'Ada' if os.path.exists('eda_output/split_train.csv') else 'Belum dibuat (jalankan EDA.py dulu)')
print('TensorFlow:', tf.__version__, '| GPU terdeteksi:', tf.config.list_physical_devices('GPU'))

## 1. Preprocessing & Ekstraksi Fitur GLCM
Dilewati otomatis bila `preprocessing_output/` sudah ada (cukup dijalankan sekali untuk semua model).

In [ ]:
import os
if not os.path.exists('preprocessing_output/preprocessed_224.npz'):
    !python run_preprocessing.py
else:
    print('Preprocessing sudah ada, dilewati.')
if os.path.exists('preprocessing_output/ringkasan_preprocessing.txt'):
    print(open('preprocessing_output/ringkasan_preprocessing.txt').read())

## 2. Pelatihan baseline M1 (ResNet50)
Tahap 1: backbone ResNet50 (bobot ImageNet) dibekukan, hanya kepala klasifikasi yang dilatih (lr 1e-3).
Tahap 2: 30 lapisan akhir dibuka untuk fine-tuning (lr 1e-5). Early stopping (patience 5) dan bobot terbaik berdasarkan `val_loss`.

In [ ]:
!python train_model.py --model M1

## 3. Pelatihan model usulan M2 (ResNet50-GLCM)
Parameter sama dengan M1; bedanya penambahan 32 fitur tekstur GLCM yang digabung sebelum lapisan klasifikasi dense.

In [ ]:
!python train_model.py --model M2

## 4. Hasil dan perbandingan M1 vs M2

In [ ]:
import json, os, pandas as pd
from IPython.display import Image, display

keys = ['akurasi_uji','precision_macro','recall_macro','f1_macro','epoch_total','epoch_terbaik',
        'val_akurasi_pada_epoch_terbaik','waktu_latih_menit','waktu_inferensi_ms_per_citra','jumlah_parameter']
models_done = [m for m in ['M1','M2'] if os.path.exists(f'hasil_{m}/ringkasan_hasil.json')]
if models_done:
    hasil = {m: json.load(open(f'hasil_{m}/ringkasan_hasil.json')) for m in models_done}
    tabel = pd.DataFrame({m: {k: r.get(k, '-') for k in keys} for m, r in hasil.items()})
    display(tabel)
    tabel.to_csv('perbandingan_M1_M2.csv')
    for m, r in hasil.items():
        print(f'\n=== {m} ({r["nama"]}) ===')
        print('Kelas paling sering tertukar:', r.get('pasangan_kelas_tertukar_terbanyak', '-'))
        if os.path.exists(f'hasil_{m}/kurva_pelatihan.png'):
            display(Image(f'hasil_{m}/kurva_pelatihan.png'))
        if os.path.exists(f'hasil_{m}/confusion_matrix.png'):
            display(Image(f'hasil_{m}/confusion_matrix.png'))

## 5. (Opsional) M3 MobileNetV2 dan M4 MobileNetV2-GLCM
Hapus tanda `#` lalu jalankan sel ini bila ingin melatih M3 dan M4.

In [ ]:
# !python train_model.py --model M3
# !python train_model.py --model M4